In [0]:
%python
%pip install snowflake-sqlalchemy snowflake-connector-python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%python
import snowflake.connector

conn = snowflake.connector.connect(
    user="Ananya24",
    password="TravelingToBoston2026",
    account="XIMWUQT-IJ47823",
    warehouse="COMPUTE_WH",
    database="CRIME_DB",
    schema="ANALYTICS"
)

print("✅ Connected to Snowflake!")

✅ Connected to Snowflake!


In [0]:
%python
sfOptions = {
  "sfURL": "YOURACCOUNT.snowflakecomputing.com",
  "sfUser": "YOUR_USER",
  "sfPassword": "YOUR_PASSWORD",
  "sfDatabase": "CRIME_DB",
  "sfSchema": "ANALYTICS",
  "sfWarehouse": "COMPUTE_WH",
  "sfRole": "ACCOUNTADMIN"
}


In [0]:
%python
df_silver = spark.read.table("la_crime_silver")


In [0]:
%python
df_silver.write \
  .format("snowflake") \
  .options(**sfOptions) \
  .option("dbtable", "STG_LA_CRIME_SILVER") \
  .mode("overwrite") \
  .save()


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6299172651627061>, line 6
      1 df_silver.write \
      2   .format("snowflake") \
      3   .options(**sfOptions) \
      4   .option("dbtable", "STG_LA_CRIME_SILVER") \
      5   .mode("overwrite") \
----> 6   .save()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1556, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1554     req.user_cont

In [0]:
%python
df = spark.sql(
    """
    SELECT COUNT(*) FROM workspace.damg7370.dim_date
    """
)
display(df)

COUNT(*)
1879


In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)



+----------+
|catalog   |
+----------+
|mm_catalog|
|samples   |
|system    |
|workspace |
+----------+



In [0]:
spark.sql("SHOW TABLES IN workspace.damg7370").toPandas()


,database,tableName,isTemporary
0,damg7370,bronze_employee_cdf,False
1,damg7370,demo_cust_bronze_rescue,False
2,damg7370,demo_cust_bronze_rescue_sd,False
3,damg7370,demo_cust_bronze_sd,False
4,damg7370,demo_cust_bronze_sd_addcols_p2,False
5,damg7370,demo_cust_bronze_sd_v2,False
6,damg7370,demo_cust_bronze_sd_v3,False
7,damg7370,demo_cust_silver_rescue,False
8,damg7370,demo_cust_silver_rescue_sd,False
9,damg7370,demo_cust_silver_sd,False


In [0]:
df_silver = spark.read.table("workspace.damg7370.la_crime_silver")
df_silver.show(5)


+---------+-------------------+-------------------+--------+----+---------+-----------+--------+------+--------------------+---------+--------+--------+------------+---------+--------------------+--------------+-----------+------+-----------+--------+--------+--------+--------+--------------------+--------------------+-------+---------+--------+---------+-----------+-----------+----------------+----------+------------+--------+----------+-----------+------------+------------+--------------+--------------+
|    dr_no|          date_rptd|           date_occ|time_occ|area|area_name|rpt_dist_no|part_1_2|crm_cd|         crm_cd_desc|  mocodes|vict_age|vict_sex|vict_descent|premis_cd|         premis_desc|weapon_used_cd|weapon_desc|status|status_desc|crm_cd_1|crm_cd_2|crm_cd_3|crm_cd_4|            location|        cross_street|    lat|      lon|year_occ|month_occ|quarter_occ|day_of_week|day_of_week_name|is_weekend|time_occ_int|occ_hour|occ_minute|time_bucket|violent_flag|vict_age_grp|vict_sex_

In [0]:
df_silver = spark.read.table("workspace.damg7370.la_crime_silver")
df_silver.show(5)


+---------+-------------------+-------------------+--------+----+---------+-----------+--------+------+--------------------+---------+--------+--------+------------+---------+--------------------+--------------+-----------+------+-----------+--------+--------+--------+--------+--------------------+--------------------+-------+---------+--------+---------+-----------+-----------+----------------+----------+------------+--------+----------+-----------+------------+------------+--------------+--------------+
|    dr_no|          date_rptd|           date_occ|time_occ|area|area_name|rpt_dist_no|part_1_2|crm_cd|         crm_cd_desc|  mocodes|vict_age|vict_sex|vict_descent|premis_cd|         premis_desc|weapon_used_cd|weapon_desc|status|status_desc|crm_cd_1|crm_cd_2|crm_cd_3|crm_cd_4|            location|        cross_street|    lat|      lon|year_occ|month_occ|quarter_occ|day_of_week|day_of_week_name|is_weekend|time_occ_int|occ_hour|occ_minute|time_bucket|violent_flag|vict_age_grp|vict_sex_

In [0]:
output_path = "/Volumes/workspace/damg7370/lacrime_csv"

df_silver.write.mode("overwrite").parquet(output_path)

print("✅ Silver saved to parquet in volume:", output_path)


✅ Silver saved to parquet in volume: /Volumes/workspace/damg7370/lacrime_csv


In [0]:
%sql
CREATE VOLUME workspace.damg7370.la_crime_silver_parquet;

In [0]:
output_path = "/Volumes/workspace/damg7370/la_crime_silver_parquet"

df.write \
    .mode("overwrite") \
    .parquet(output_path)

display(df)

dr_no,date_rptd,date_occ,time_occ,area,area_name,rpt_dist_no,part_1_2,crm_cd,crm_cd_desc,mocodes,vict_age,vict_sex,vict_descent,premis_cd,premis_desc,weapon_used_cd,weapon_desc,status,status_desc,crm_cd_1,crm_cd_2,crm_cd_3,crm_cd_4,location,cross_street,lat,lon,year_occ,month_occ,quarter_occ,day_of_week,day_of_week_name,is_weekend,time_occ_int,occ_hour,occ_minute,time_bucket,violent_flag,vict_age_grp,vict_sex_clean,geo_valid_flag
240104892,2024-01-13T00:00:00.000Z,2024-01-13T00:00:00.000Z,1700,1,Central,152,2,888,TRESPASSING,0910 1501,0,X,X,835,7TH AND METRO CENTER (NOT LINE SPECIFIC),null,null,IC,Invest Cont,888,null,null,null,600 S FIGUEROA ST,null,34.0508,-118.2586,2024,1,1,Sat,Saturday,Y,1700,17,0,17:00,N,Juvenile,X,Y
241911229,2024-08-27T00:00:00.000Z,2024-08-21T00:00:00.000Z,1800,19,Mission,1964,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND OVER),0385,61,F,H,101,STREET,null,null,IC,Invest Cont,331,null,null,null,KESTER AV,TUPPER ST,34.2391,-118.4562,2024,8,3,Wed,Wednesday,N,1800,18,0,18:00,Y,51+,F,Y
240808423,2024-05-15T00:00:00.000Z,2024-05-02T00:00:00.000Z,1900,8,West LA,857,1,440,THEFT PLAIN - PETTY ($950 & UNDER),0344,37,M,A,116,OTHER/OUTSIDE,null,null,IC,Invest Cont,440,null,null,null,1200 EDRIS DR,null,34.0571,-118.398,2024,5,2,Thu,Thursday,N,1900,19,0,19:00,Y,31-50,M,Y
240913841,2024-12-28T00:00:00.000Z,2024-12-28T00:00:00.000Z,425,9,Van Nuys,994,1,330,BURGLARY FROM VEHICLE,0344 1822,44,F,W,104,DRIVEWAY,null,null,IC,Invest Cont,330,null,null,null,3500 CAMINO DE LA CUMBRE,null,34.1345,-118.4463,2024,12,4,Sat,Saturday,Y,425,4,25,4.:00,Y,31-50,F,Y
241610828,2024-10-12T00:00:00.000Z,2024-10-11T00:00:00.000Z,1800,16,Foothill,1605,1,510,VEHICLE - STOLEN,null,0,null,null,101,STREET,null,null,IC,Invest Cont,510,null,null,null,11900 KATHYANN ST,null,34.2847,-118.3921,2024,10,4,Fri,Friday,N,1800,18,0,18:00,Y,Juvenile,X,Y
241009278,2024-06-06T00:00:00.000Z,2024-06-06T00:00:00.000Z,740,10,West Valley,1033,1,440,THEFT PLAIN - PETTY ($950 & UNDER),1822 0344,36,M,H,510,NURSING/CONVALESCENT/RETIREMENT HOME,null,null,IC,Invest Cont,440,null,null,null,6400 WILBUR AV,null,34.1866,-118.5447,2024,6,2,Thu,Thursday,N,740,7,40,7.:00,Y,31-50,M,Y
240118725,2024-09-19T00:00:00.000Z,2024-09-19T00:00:00.000Z,2150,1,Central,132,2,888,TRESPASSING,0910 1501,0,X,X,135,MTA PROPERTY OR PARKING LOT,null,null,IC,Invest Cont,888,null,null,null,200 S HOPE ST,null,34.0549,-118.2511,2024,9,3,Thu,Thursday,N,2150,21,50,21:00,N,Juvenile,X,Y
240115485,2024-07-27T00:00:00.000Z,2024-07-14T00:00:00.000Z,854,1,Central,129,1,440,THEFT PLAIN - PETTY ($950 & UNDER),0910 0344 0329,0,X,X,135,MTA PROPERTY OR PARKING LOT,null,null,IC,Invest Cont,440,null,null,null,700 E TEMPLE ST,null,34.0502,-118.2336,2024,7,3,Sun,Sunday,Y,854,8,54,8.:00,Y,Juvenile,X,Y
241806640,2024-02-27T00:00:00.000Z,2024-02-26T00:00:00.000Z,840,18,Southeast,1891,1,440,THEFT PLAIN - PETTY ($950 & UNDER),0352 1822 0344,42,F,O,501,SINGLE FAMILY DWELLING,null,null,IC,Invest Cont,440,null,null,null,16900 S DENVER AV,null,33.8785,-118.2836,2024,2,1,Mon,Monday,N,840,8,40,8.:00,Y,31-50,F,Y
240812407,2024-12-09T00:00:00.000Z,2024-12-06T00:00:00.000Z,1800,8,West LA,825,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),null,0,null,null,101,STREET,null,null,IC,Invest Cont,420,null,null,null,12200 DUNOON LN,null,34.0536,-118.4763,2024,12,4,Fri,Friday,N,1800,18,0,18:00,Y,Juvenile,X,Y
